# 02 Data cleaning – Rama & Uyazi
**Goal:** turn the raw 2009 H1N1 Flu Survey data into a clean, model-ready table with steps that can be re-used on the test set.

This notebook combines both of our cleaning notebooks. All logic lives in `src/cleaning.py`; the notebook shows the evidence for each decision and calls the functions.
Parts marked **(Uyazi)** come from Uyazi's notebook, parts marked **(Rama)** from Rama's. See `reports/cleaning_changes.md` for what changed and why.

**Rules:** (1) never change the raw data, (2) check before you fill, (3) anything *learned* from data (common job codes, most frequent values) is learned from the **training** set only and then applied to the test set.

Tickets: 1.1 (quality), 1.2 (missing values), 1.7 (high-cardinality columns), 1.9 (cleaning code).

In [ ]:
import sys
sys.path.append("..")
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.cleaning import (TARGETS, tidy_text, check_ranges, validate, rare_levels,
                          clean, learn_modes, fill_numeric)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

train_X = pd.read_csv("../data/training_set_features.csv")
train_y = pd.read_csv("../data/training_set_labels.csv")
test_X = pd.read_csv("../data/test_set_features.csv")

raw = train_X.merge(train_y, on="respondent_id", validate="one_to_one")   # raw training data
print("train:", raw.shape, "| test:", test_X.shape)

## 1. Quality checks (tickets 1.1)

In [ ]:
print("Respondent ids unique:", raw["respondent_id"].is_unique)
print("Duplicate answer rows :", raw.drop(columns="respondent_id").duplicated().sum())
print("Missing in the targets :", raw[TARGETS].isnull().sum().to_dict())
print("Values outside the expected range:")
check_ranges(raw)

## 2. Tidy text (Uyazi)
`census_msa` contains a double space (`"MSA, Not Principle  City"`). `tidy_text` strips and collapses whitespace so categories are never split by invisible differences.

In [ ]:
print("before:", sorted(raw["census_msa"].dropna().unique()))
print("after :", sorted(tidy_text(raw)["census_msa"].dropna().unique()))

## 3. Missing values (ticket 1.2)

In [ ]:
missing = raw.isnull().mean().mul(100).sort_values(ascending=False)
missing = missing[missing > 0]
ax = missing.sort_values().plot.barh(figsize=(7, 7), color="#4f81bd")
ax.set_xlabel("% missing"); ax.set_title("Missing values per column")
plt.tight_layout(); plt.show()

### Is *missing* related to vaccination? (Rama)
If vaccination rates differ between people who skipped a question and people who answered it, the gap itself carries information and must not be erased.

In [ ]:
rows = []
for col in missing.index:
    m = raw[col].isnull()
    rows.append({"column": col, "% missing": round(m.mean() * 100, 1),
                 "h1n1 if missing": round(raw.loc[m, "h1n1_vaccine"].mean(), 3),
                 "h1n1 if answered": round(raw.loc[~m, "h1n1_vaccine"].mean(), 3),
                 "seasonal if missing": round(raw.loc[m, "seasonal_vaccine"].mean(), 3),
                 "seasonal if answered": round(raw.loc[~m, "seasonal_vaccine"].mean(), 3)})
pd.DataFrame(rows)

**Decisions**
| Column(s) | Finding | Decision |
|---|---|---|
| Columns with > 5 % missing **(Uyazi's rule)** | Skipped questions can be informative | Add a `<col>_missing` flag *before* filling |
| `health_insurance` (46 %) | H1N1 uptake 11 % if missing vs 30 % if answered | Flag; fill only for the model, the flag keeps the signal |
| `household_*` (1 %) **(Rama)** | Seasonal uptake about 26 % if missing vs 47 % | Flag as well, although they are below 5 % |
| `employment_industry/occupation` | Missing mostly because the person does not work | `Not employed` (see section 4), no extra flag needed |
| Remaining text gaps | Small | Category `Missing` |
| Remaining numeric gaps | 0.1–8 % | Most frequent value from the **training** set (flags keep the information); in Sprint 2 move this into a Pipeline |

## 4. Employment columns

In [ ]:
no_industry = raw["employment_industry"].isnull()
print(raw.loc[no_industry, "employment_status"].value_counts(dropna=False))

10,231 are *Not in Labor Force* and 1,453 *Unemployed* → `Not employed` (Uyazi's label). 1,463 have no employment status at all → `Missing`. Only 183 employed people are really missing an industry.

## 5. High-cardinality columns (ticket 1.7) (Rama)

In [ ]:
def level_table(col):
    s = raw[col].fillna("(missing)")
    t = raw.groupby(s)[TARGETS].mean().round(3)
    t["n"] = s.value_counts()
    t["% of people"] = (t["n"] / len(raw) * 100).round(1)
    return t.sort_values("n", ascending=False)

level_table("employment_industry")

In [ ]:
level_table("employment_occupation")

In [ ]:
level_table("hhs_geo_region")

In [ ]:
a = raw["employment_industry"] == "haxffmxo"
b = raw["employment_occupation"] == "dcjcmpih"
print("industry haxffmxo:", a.sum(), "| occupation dcjcmpih:", b.sum(), "| same people:", (a & b).sum())

**Decisions**
- Grouping by frequency alone is a bad rule: `haxffmxo` / `dcjcmpih` are small (148 people, the **same** people) but have a 62 % H1N1 uptake.
- Only codes with fewer than 100 people become `Other`. The list is learned from the **training** set (`rare_levels`) and re-used for the test set.
- `hhs_geo_region` stays as it is (10 regions, 18–26 % H1N1 uptake).

## 6. Apply the cleaning to train **and** test

In [ ]:
keep = rare_levels(tidy_text(raw))              # learned from TRAIN only
train_c = clean(raw, keep)                      # text, flags, job codes, ordered columns
modes = learn_modes(train_c)                    # most frequent values from TRAIN only
train_clean = fill_numeric(train_c, modes)
test_clean = fill_numeric(clean(test_X, keep), modes)

print("train:", train_clean.shape, "| test:", test_clean.shape)
print("job codes kept:", {c: len(v) for c, v in keep.items()})

## 7. Validation (Uyazi's assertions, now reusable functions)

In [ ]:
validate(train_clean, raw=raw)      # no gaps, ids unique, ranges ok, rows and targets unchanged
validate(test_clean)                # same rules for the test set
print("All checks passed.")
print("Same feature columns in train and test:",
      list(train_clean.drop(columns=TARGETS).columns) == list(test_clean.columns))

In [ ]:
# Cleaning must not change the vaccination rates
pd.DataFrame({"raw %": raw[TARGETS].mean().mul(100).round(1),
              "clean %": train_clean[TARGETS].mean().mul(100).round(1)})

In [ ]:
bins = pd.cut(train_clean["n_missing_answers"], [-1, 0, 1, 2, 4, 30])
train_clean.groupby(bins, observed=True)[TARGETS].agg(["mean", "count"]).round(3)

## 8. Save

In [ ]:
out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)
train_clean.set_index("respondent_id").to_csv(out / "train_clean.csv")     # id as index (Uyazi)
test_clean.set_index("respondent_id").to_csv(out / "test_clean.csv")
print("saved to", out.resolve())

CSV files are ignored by Git, so teammates re-create them by running this notebook.

In [ ]:
new_cols = [c for c in train_clean.columns if c not in raw.columns]
pd.DataFrame({"new column": new_cols, "dtype": [str(train_clean[c].dtype) for c in new_cols]})

## Decision log (copy to `workflow.md`)
| Decision | Reason | From |
|---|---|---|
| Strip / collapse whitespace in text | `census_msa` had a double space | Uyazi |
| Flag every column with > 5 % missing (plus `household_*`) before filling | Missingness is related to vaccination | Uyazi + Rama |
| `Not employed` / `Missing` labels | Clear names, one convention | Uyazi |
| Job codes < 100 people → `Other`, larger rare ones kept | Rare codes can be very different (62 % uptake) | Rama |
| Ordered columns `age_group_ord`, `education_ord`, `income_ord` | Keeps the order for linear models | Rama |
| Numeric gaps filled with training-set modes | Small gaps; flags keep the signal | Uyazi (made reusable by Rama) |
| `validate()` after cleaning | Catches silent errors early | Uyazi |
| Fill values learned from train only, then applied to test | No leakage; same rules for new data | Rama |
| Sprint 2: move the mode fill into a Pipeline | Re-learned inside every CV fold | both |